# 01 · Keşifsel veri analizi (EDA)

**Amaç:** Faz 3'te kuracağımız modellerin varsayımlarını gerçek veriyle sınamak. Model kurmadan önce şu sorulara cevap arıyoruz:

1. Goller **Poisson** dağılımına uyuyor mu?
2. Ev ve deplasman golleri **bağımsız** mı? Düşük skorlarda (0-0, 1-1) sapma var mı? Varsa **Dixon-Coles** düzeltmesi gerekir.
3. Kornerler **aşırı dağılımlı** mı? Öyleyse Poisson yerine **Negatif Binom** kullanmalıyız.
4. Ev sahibi avantajı ligden lige ve sezondan sezona ne kadar değişiyor?
5. Bahis piyasası ne kadar iyi tahmin ediyor? Bu, modelimizin geçmesi gereken **çıta**.

**Veri:** football-data.co.uk, 5 lig, 2021/22'den bugüne (gerçek veri, CI hattında indirilir).
Hesaplar `footintel.analysis.eda` modülünde; bu notebook yalnızca çağırır ve çizer. Böylece hesaplar test edilebilir.

> Understat (xG) bu çalıştırmada erişilemiyorsa xG bölümü atlanır; xG analizi kaynak geri geldiğinde eklenecek.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from footintel.analysis import eda
from footintel.config import CLEAN_DIR, LEAGUES

pd.set_option("display.precision", 3)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
NAMES = {k: v.name for k, v in LEAGUES.items()}

fd = eda.played(pd.read_parquet(CLEAN_DIR / "matches_football_data.parquet"))
fd["total_corners"] = fd["home_corners"] + fd["away_corners"]
fd["league_name"] = fd["league"].map(NAMES)
print(f"{len(fd):,} oynanmış maç")
fd.pivot_table(index="league_name", columns="season_label", values="match_id", aggfunc="count")

## 1. Goller Poisson'a uyuyor mu?

**Poisson dağılımı** "belirli bir sürede, birbirinden bağımsız ve sabit hızla gerçekleşen olay sayısı"nı modeller. Tek parametresi vardır: ortalama λ. Önemli özelliği **varyans = ortalama** olmasıdır.

Bu yüzden ilk test basit: **varyans / ortalama** oranı.
- ≈ 1 → Poisson uygun.
- belirgin şekilde > 1 → *aşırı dağılım* (overdispersion), Negatif Binom düşünülmeli.
- < 1 → *eksik dağılım*, değerler Poisson'un söylediğinden daha toplu.

Not: Lig genelindeki oran, farklı güçteki takımları karıştırdığı için 1'in biraz üstünde çıkabilir. Faz 3'te takım gücünü modele koyunca kalan dağılıma tekrar bakacağız.

In [ ]:
disp = eda.dispersion(fd, ["home_goals", "away_goals", "total_goals", "total_corners"])
disp.assign(league=disp["league"].map(NAMES)).pivot(
    index="league", columns="metric", values=["mean", "var_mean_ratio"]
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, col, title in [
    (axes[0], "home_goals", "Ev sahibi golleri"),
    (axes[1], "away_goals", "Deplasman golleri"),
]:
    t = eda.poisson_fit(fd[col], max_k=6)
    x = np.arange(len(t))
    ax.bar(x - 0.2, t["observed"], width=0.4, label="Gözlenen")
    ax.bar(x + 0.2, t["poisson"], width=0.4, label="Poisson (aynı ortalama)")
    ax.set_xticks(x, t["goals"])
    ax.set_title(title)
    ax.set_xlabel("gol")
axes[0].set_ylabel("maç oranı")
axes[0].legend(frameon=False)
plt.tight_layout()

## 2. Ev sahibi avantajı

Elo modelimizde ev avantajını tek bir sabit (50 puan) aldık. Ligler arası fark büyükse Faz 3'te **lig bazında** ev avantajı parametresi gerekir.

In [ ]:
ha = eda.home_advantage(fd)
ha["league"] = ha["league"].map(NAMES)
fig, ax = plt.subplots(figsize=(9, 3.8))
for lg, g in ha.groupby("league"):
    ax.plot(g["season"], g["home_win"], marker="o", label=lg)
ax.set_ylabel("ev sahibi galibiyet oranı")
ax.legend(frameon=False, ncol=3)
ax.set_title("Ev sahibi galibiyet oranı, sezon bazında")
plt.tight_layout()
ha.pivot(index="league", columns="season", values="goal_diff")

## 3. Ev ve deplasman golleri bağımsız mı? (Dixon-Coles'un gerekçesi)

Basit Poisson modeli iki takımın gollerini **bağımsız** sayar: P(1-1) = P(ev 1 gol) × P(dep 1 gol).
Dixon ve Coles (1997), gerçek maçlarda özellikle **0-0 ve 1-1**'in bu çarpımdan sık, **1-0 ve 0-1**'in ise seyrek görüldüğünü gösterdi ve bu dört skoru düzelten bir ρ (rho) parametresi ekledi.

Aşağıda her skor için **gözlenen oran / bağımsızlıkta beklenen oran** var. 1'in üstü = beklenenden sık.

**Dikkat (yanıltıcı olabilir):** Burada takım gücünü kontrol etmiyoruz. Güçlü-zayıf eşleşmelerin karışımı da bağımlılık gibi görünebilir. Kesin cevabı Faz 3'te ρ'yu takım güçleriyle birlikte tahmin ederek alacağız.

In [ ]:
ls = eda.low_score_check(fd)
ls["league"] = ls["league"].map(NAMES)
tab = ls.pivot(index="league", columns="score", values="ratio")[[f"{i}-{j}" for i, j in eda.SCORES]]
fig, ax = plt.subplots(figsize=(8, 3.4))
im = ax.imshow(tab.to_numpy(), cmap="RdBu_r", vmin=0.8, vmax=1.2)
ax.set_xticks(range(tab.shape[1]), tab.columns)
ax.set_yticks(range(tab.shape[0]), tab.index)
for i in range(tab.shape[0]):
    for j in range(tab.shape[1]):
        ax.text(j, i, f"{tab.iat[i, j]:.2f}", ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=ax, label="gözlenen / beklenen")
ax.set_title("Skor sıklığı: gözlenen / bağımsızlıkta beklenen")
plt.tight_layout()

In [ ]:
fd.groupby("league_name")[["home_goals", "away_goals"]].corr().unstack().iloc[:, 1].rename(
    "ev-dep gol korelasyonu"
)

## 4. Kornerler: Poisson mu, Negatif Binom mu?

**Negatif Binom**, Poisson'un "varyansı ortalamadan büyük olabilen" genellemesidir. Ek bir parametre (dağılım parametresi) taşır. Kornerler maçın akışına (skor durumu, baskı) çok bağlı olduğundan aşırı dağılım beklenir; aşağıdaki oran bunu sınıyor.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
c = fd["total_corners"].dropna().astype(int)
counts = c.value_counts(normalize=True).sort_index()
lam = c.mean()
k = np.arange(0, c.max() + 1)
ax.bar(counts.index, counts.values, label="Gözlenen")
ax.plot(
    k,
    [eda.poisson_pmf(int(i), lam) for i in k],
    "o-",
    color="black",
    ms=3,
    label=f"Poisson (λ={lam:.1f})",
)
ax.set_xlabel("maçtaki toplam korner")
ax.set_ylabel("maç oranı")
ax.legend(frameon=False)
ax.set_title(f"Toplam korner (varyans/ortalama = {c.var() / lam:.2f})")
plt.tight_layout()

## 5. Bahis pazarları ne sıklıkla gerçekleşiyor?

Bu oranlar modelin "ön bilgisi"dir: Örneğin bir ligde 2.5 üst oranı %50 ise, model hiçbir şey bilmeden %50 diyebilir. Modelin değeri bu ortalamadan **maç bazında** sapabilmesindedir.

In [ ]:
eda.market_rates(fd).rename(index=NAMES)

## 6. Çıta: bahis piyasası ne kadar iyi?

Bahis oranlarından olasılık çıkarırız: **1 / oran**. Ama üç sonucun toplamı 1'den büyük çıkar; aradaki fark bahis şirketinin **marjıdır** (overround). Toplamı 1'e normalize ederek "piyasanın olasılığını" buluruz.

İki ölçüt kullanıyoruz (Faz 5'teki tahmin defterinde de bunlar olacak):
- **Brier skoru:** Tahmin olasılıkları ile gerçekleşen sonuç (1/0) arasındaki farkların karesi, üç sonuç üzerinden toplanır. 0 mükemmel; her maça 1/3-1/3-1/3 demek ≈ 0.667.
- **Log loss:** Gerçekleşen sonuca verilen olasılığın eksi logaritması. "%5 ihtimal" dediğin sonuç olursa ağır cezalandırır.

**Kapanış oranları** (maç başlamadan hemen önceki) piyasanın tüm bilgiyi topladığı noktadır ve genelde en zor yenilen çıtadır.

In [ ]:
bench = pd.concat(
    {
        "kapanış": eda.market_benchmark(fd, "close").set_index("league"),
        "açılış/ortalama": eda.market_benchmark(fd, "avg").set_index("league"),
    },
    axis=1,
).rename(index=NAMES)
bench

### Kalibrasyon

Brier tek bir sayı verir. **Kalibrasyon** ise "piyasa %60 dediğinde gerçekten %60 mı oluyor?" sorusunu cevaplar. Noktalar köşegene yakınsa olasılıklar güvenilirdir.

In [ ]:
p = eda.implied_1x2(fd)
fig, ax = plt.subplots(figsize=(4.6, 4.6))
for col, outcome, label in [
    ("p_home", fd["home_goals"] > fd["away_goals"], "Ev"),
    ("p_draw", fd["home_goals"] == fd["away_goals"], "Beraberlik"),
    ("p_away", fd["home_goals"] < fd["away_goals"], "Deplasman"),
]:
    t = eda.calibration_table(p[col], outcome)
    t = t[t["n"] >= 30]
    ax.plot(t["mean_pred"], t["observed"], "o-", label=label)
ax.plot([0, 1], [0, 1], "--", color="grey", lw=1)
ax.set_xlabel("piyasanın olasılığı")
ax.set_ylabel("gerçekleşme oranı")
ax.legend(frameon=False)
ax.set_title("Piyasa kalibrasyonu (kapanış)")
plt.tight_layout()

## 7. Kendi Elo modelimiz piyasaya ne kadar yakın?

Elo yalnızca "beklenen ev skoru" verir (galibiyet 1, beraberlik 0.5). Piyasayı aynı ölçeğe çevirmek için: p_ev + 0.5 × p_beraberlik. Böylece ikisini 2025/26 test sezonunda aynı MSE ile karşılaştırabiliyoruz.

In [ ]:
elo_path = CLEAN_DIR / "match_elo.parquet"
eda.elo_vs_market(
    fd, pd.read_parquet(elo_path), ["2526"]
) if elo_path.exists() else "match_elo.parquet yok: önce pipelines/build_elo.py"

## Bulgular ve Faz 3'e etkisi

*Aşağıdaki sayılar 2021/22 – 2026/27 (4 Kasım 2026'ya kadar) gerçek verisinden, CI çalıştırmasından alındı. Veri güncellendikçe yukarıdaki tablolar değişir; bu yorumlar o tarihe aittir.*

**Veri kontrolü.** Ligue 1'de 2023/24'ten sonra 306 maç var (lig 18 takıma indi). Süper Lig'de 2022/23'te 342 maç var, çünkü deprem nedeniyle iki takım ligden çekildi. 2024/25'te de 342 maç var, çünkü lig 19 takımlıydı ve her hafta bir takım bay geçti. Yani eksik görünen maçlar veri hatası değil, gerçek lig yapısı.

1. **Goller Poisson'a iyi uyuyor.** Varyans/ortalama oranı ev ve deplasman golleri için 1.02–1.12, toplam gol için 0.95–1.05. Gözlenen ve Poisson frekansları her gol sayısında 1 puandan az farklı. Küçük aşırı dağılım, takım güçlerinin karışmasından bekleniyor. **Karar:** Faz 3'te gol modeli Poisson tabanlı olacak.

2. **Düşük skorlarda sapma var ama karışık.** 0-1 her ligde beklenenden seyrek (0.84–0.95), 1-1 beş ligin dördünde sık (1.04–1.08). Bu, Dixon-Coles'un öngördüğü yönde. Ama 0-0 Premier League'de beklenenden belirgin şekilde seyrek (0.84), Süper Lig ve Ligue 1'de ise sık. Ev-deplasman gol korelasyonu da hafif negatif (–0.13 ile 0.01 arası). **Karar:** ρ'yu sabit varsaymayacağız, Faz 3'te veriden (lig bazında) tahmin edip sıfırdan anlamlı şekilde farklı mı diye sınayacağız.

3. **Ev avantajı ligden lige ve sezondan sezona değişiyor.** Ev sahibi galibiyet oranı Serie A'da %39–47, Süper Lig'de %42–49 arasında. **Karar:** Faz 3'te ev avantajı lig bazında bir parametre olacak.

4. **Kornerlerde aşırı dağılım beklenenden az.** Varyans/ortalama 1.11–1.23 (en yüksek Serie A). Poisson'dan sapma var ama küçük. **Karar:** Negatif Binom ve Poisson'u Faz 3'te karşılaştıracağız. Fark küçükse basit modeli seçeceğiz.

5. **Pazar oranları liglere göre farklı.** 2.5 üst oranı La Liga'da %48, Premier League'de %56. Karşılıklı gol %52–55. Her lig için ayrı ortalama gol seviyesi gerekiyor.

6. **Çıta: piyasa.** Kapanış oranlarıyla 1X2 Brier skoru 0.567–0.585 (her maça 1/3 demek 0.667), log loss 0.957–0.982. Kapanış oranları açılıştan her ligde biraz daha iyi. En zor lig Ligue 1, bahis marjı en yüksek lig Süper Lig (%6.6).

7. **Piyasada favori-sürpriz yanlılığı görünüyor.** Piyasanın %8 dediği ev galibiyetleri %2, %16 dediği galibiyetler %13 oranında gerçekleşiyor. Yüksek olasılıklarda ise tersi oluyor. Basit (orantılı) normalizasyon marjı tüm sonuçlara eşit dağıtıyor, oysa bahisçiler marjı sürprizlere daha fazla yüklüyor. **Karar:** Faz 5'te Shin yöntemiyle düzelteceğiz.

8. **Elo, piyasaya yakın ama geride.** 2025/26'da beklenen ev skoru MSE'si Elo'da 0.158, kapanış piyasasında 0.149. Yalnızca sonuçlardan beslenen tek bir sayının bu kadar yaklaşması iyi bir başlangıç. Aradaki farkı kapatmak için gol, şut ve korner bilgisini kullanan modeller Faz 3'ün işi.